In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(val_preds)))
print('Validation RMSE:', rmse)

NameError: name 'y_val' is not defined

In [ ]:
df = pd.read_csv('train.csv')
df.head()

In [ ]:
df.shape
df.info()
df.isnull().sum()

In [ ]:
test = pd.read_csv('test.csv')
test.head()

In [ ]:
df['product_category'] = df['product_category'].str.lower()
test['product_category'] = test['product_category'].str.lower()

In [ ]:
print(df['product_category'].unique())

In [ ]:
print(test['product_category'].unique())

In [ ]:
df['store_size'] = df['store_size'].fillna('Unknown')
test['store_size'] = test['store_size'].fillna('Unknown')

In [ ]:
print(df['store_size'].value_counts())

In [ ]:
df['product_weight_kg'] = df['product_weight_kg'].fillna(df.groupby('product_category')['product_weight_kg'].transform('median'))
test['product_weight_kg'] = test['product_weight_kg'].fillna(test.groupby('product_category')['product_weight_kg'].transform('median'))

In [ ]:
df.isnull().sum()
test.isnull().sum()

In [ ]:
sns.histplot(df['total_sales'], kde=True)
plt.show()

In [ ]:
df['log_total_sales'] = np.log1p(df['total_sales'])

In [ ]:
sns.histplot(df['log_total_sales'], kde=True)
plt.show()

In [ ]:
# Combine train and test temporarily so encoding is consistent across both
df['is_train'] = 1
test['is_train'] = 0
test['total_sales'] = np.nan  # placeholder, test has no target
test['log_total_sales'] = np.nan

combined = pd.concat([df, test], sort=False)

# One-hot encode categorical columns
categorical_cols = ['fat_content', 'product_category', 'store_size', 'store_location_tier', 'store_format']
combined = pd.get_dummies(combined, columns=categorical_cols, drop_first=True)

# Split back into train/test
df_encoded = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_encoded = combined[combined['is_train'] == 0].drop(columns=['is_train', 'total_sales', 'log_total_sales'])

In [ ]:
feature_cols = [c for c in df_encoded.columns if c not in ['id', 'product_code', 'store_code', 'total_sales', 'log_total_sales']]

X = df_encoded[feature_cols]
y = df_encoded['log_total_sales']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(val_preds)))
print('Validation RMSE:', rmse)

In [ ]:
test_preds_log = model.predict(test_encoded[feature_cols])
test_preds = np.expm1(test_preds_log)  # reverse the log1p transform

submission = pd.DataFrame({
    'id': test_encoded['id'],
    'total_sales': test_preds
})
submission.to_csv('submission.csv', index=False)

In [ ]:
import xgboost as xgb
print(xgb.__version__)

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

xgb_val_preds = xgb_model.predict(X_val)
xgb_rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(xgb_val_preds)))
print('XGBoost Validation RMSE:', xgb_rmse)

In [ ]:
xgb_test_preds_log = xgb_model.predict(test_encoded[feature_cols])
xgb_test_preds = np.expm1(xgb_test_preds_log)

submission = pd.DataFrame({
    'id': test_encoded['id'],
    'total_sales': xgb_test_preds
})
submission.to_csv('submission.csv', index=False)

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_rmses = []
oof_preds = np.zeros(len(X))  # out-of-fold predictions, one per training row

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y.iloc[train_idx], y.iloc[val_idx]

    fold_model = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
    fold_model.fit(X_tr, y_tr)

    fold_preds = fold_model.predict(X_vl)
    oof_preds[val_idx] = fold_preds

    fold_rmse = np.sqrt(mean_squared_error(np.expm1(y_vl), np.expm1(fold_preds)))
    fold_rmses.append(fold_rmse)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.2f}')

overall_rmse = np.sqrt(mean_squared_error(np.expm1(y), np.expm1(oof_preds)))
print(f'\nMean fold RMSE: {np.mean(fold_rmses):.2f}')
print(f'Overall OOF RMSE: {overall_rmse:.2f}')

In [ ]:
final_model = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
final_model.fit(X, y)

final_test_preds_log = final_model.predict(test_encoded[feature_cols])
final_test_preds = np.expm1(final_test_preds_log)

submission = pd.DataFrame({
    'id': test_encoded['id'],
    'total_sales': final_test_preds
})
submission.to_csv('submission.csv', index=False)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

search = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X, y)

print('Best params:', search.best_params_)
print('Best CV RMSE (log-space):', -search.best_score_)

In [ ]:
tuned_model = XGBRegressor(
    subsample=0.9,
    n_estimators=400,
    max_depth=4,
    learning_rate=0.01,
    colsample_bytree=1.0,
    random_state=42,
    n_jobs=-1
)

tuned_model.fit(X, y)

# Quick 5-fold CV check with the tuned params, for a fair comparison to your earlier 1132.68
kf = KFold(n_splits=5, shuffle=True, random_state=42)
tuned_oof = np.zeros(len(X))

for train_idx, val_idx in kf.split(X):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y.iloc[train_idx], y.iloc[val_idx]

    m = XGBRegressor(subsample=0.9, n_estimators=400, max_depth=4, learning_rate=0.01, colsample_bytree=1.0, random_state=42, n_jobs=-1)
    m.fit(X_tr, y_tr)
    tuned_oof[val_idx] = m.predict(X_vl)

tuned_rmse = np.sqrt(mean_squared_error(np.expm1(y), np.expm1(tuned_oof)))
print('Tuned model OOF RMSE:', tuned_rmse)
print('Previous (untuned) OOF RMSE: 1132.68')

In [ ]:
tuned_test_preds_log = tuned_model.predict(test_encoded[feature_cols])
tuned_test_preds = np.expm1(tuned_test_preds_log)

submission = pd.DataFrame({
    'id': test_encoded['id'],
    'total_sales': tuned_test_preds
})
submission.to_csv('submission.csv', index=False)